In [1]:
import json
import pandas as pd
import re

with open('games.json', encoding="utf8") as file:
    d=json.loads(file.read())

new_games_df = pd.DataFrame.from_dict(d).T.drop(['affiliate_links', 'backloggd_link', 'content', 'submitted_by', 'twitter_ids'], axis=1)
new_games_df["game"] = new_games_df['answers'].apply(lambda x: x[0])
new_games_df = new_games_df.drop(['answers'], axis=1)

def clear_platforms(x):
    x = re.sub(r'\(.*\)', '', x)
    if x.startswith('2020: PC'):
        return ['PC']
    x = x.replace(".", ",")
    x = x.split(",")
    x = [s.strip() for s in x]
    x = ["PS1" if s == "PS" else s for s in x]
    x = ["X360" if s == "360" else s for s in x]
    x = ["GameCube" if s == "Gamecube" else s for s in x]
    x = ["Xbox Series" if s == "X Series X/S" else s for s in x]
    x = ["Xbox Series" if s == "Series X/S" else s for s in x]
    x = ["Xbox Series" if s == "Xbox series" else s for s in x]
    x = ["Mac" if s == "Macintosh" else s for s in x]
    return x

new_games_df["console_platform"] = new_games_df["console_platform"].apply(clear_platforms)

def clear_genres(x):
    x = re.sub(r'\(.*\)', '', x)
    x = x.replace(".", ",")
    x = x.split(",")
    x = [s.strip().casefold() for s in x]
    return x

new_games_df["genre"] = new_games_df["genre"].apply(clear_genres)

def clear_metacritic(x):
    if x and x[0].isalpha():
        return None
    x = re.search(r"[0-9]+", x)
    return x.group(0)

new_games_df["metacritic_score"] = new_games_df["metacritic_score"].apply(clear_metacritic)
new_games_df["metacritic_score"] = new_games_df["metacritic_score"].astype(float)

new_games_df["release_year"] = new_games_df["release_year"].apply(lambda x: re.search(r"[0-9]{4}", x).group(0))
new_games_df["release_year"] = new_games_df["release_year"].astype(int)

new_games_df["day"] = new_games_df.index.astype(int)

new_games_df.head(10)

,console_platform,developer,franchise,genre,metacritic_score,release_year,game,day
1,"[PC, PS3, X360]",Bioware,Mass Effect,"[action, rpg]",89.0,2012,Mass Effect 3,1
2,[N64],Nintendo,The Legend of Zelda,"[action, adventure]",99.0,1998,The Legend of Zelda: Ocarina of Time,2
3,"[PC, PS4, XONE, Switch]",Mobius Digital,None,"[open world, puzzle]",85.0,2019,Outer Wilds,3
4,[PS4],Guerrilla Games,Horizon,"[action, rpg]",89.0,2017,Horizon Zero Dawn,4
5,[PC],LucasArts,Monkey Island,"[point & click, adventure]",NaN,1990,The Secret of Monkey Island,5
6,[PS2],Rockstar Games,Grand Theft Auto,"[action, adventure]",93.0,2001,Grand Theft Auto 3,6
7,[GameCube],Capcom Production Studio 4,Resident Evil,"[survival, horror, action, adventure]",96.0,2005,Resident Evil 4,7
8,"[PC, PS3, X360]",Valve,Portal,[puzzle],95.0,2011,Portal 2,8
9,[PS2],SCE Japan Studio,Ico,"[action, adventure]",91.0,2005,Shadow of the Colossus,9
10,[N64],Nintendo,Mario,[sports],91.0,1999,Mario Golf,10


In [ ]:
len()

In [5]:
new_games_df[new_games_df['console_platform'].apply(lambda x: 'Genesis' in x or 'Dreamcast' in x)]

,console_platform,developer,franchise,genre,metacritic_score,release_year,game,day
177,[Genesis],Sonic Team,Sonic The Hedgehog,[platformer],NaN,1991,Sonic the Hedgehog,177
347,[Dreamcast],Sonic Team USA,Sonic The Hedgehog,"[platform, action, adventure]",89.0,2001,Sonic Adventure 2,347
418,[Dreamcast],Smilebit,Jet Set Radio,[action],94.0,2000,Jet Set Radio,418
502,[Dreamcast],Sonic Team,Sonic The Hedgehog,[action],NaN,1999,Sonic Adventure,502
524,"[PS1, PC, Dreamcast]","Neversoft Entertainment, Inc.",Tony Hawk,[sports],91.0,2000,Tony Hawk's Pro Skater 2,524
706,"[DOS, SNES, Amiga, Genesis]","Silicon & Synapse, Inc.",The Lost Vikings,"[puzzle, platformer]",NaN,1993,The Lost Vikings,706
824,"[PC, PS1, Dreamcast]",Core Design,Tomb Raider,"[action, adventure, platform]",NaN,1999,Tomb Raider: The Last Revelation,824
859,[Dreamcast],Overworks,Skies of Arcadia,"[jrpg, adventure]",93.0,2000,Skies of Arcadia,859
876,"[Arcade, Dreamcast, PS2]",Team Ninja,Dead or Alive,[fighting],NaN,1999,Dead or Alive 2,876
894,[Genesis],Sega Technical Institute,Sonic The Hedgehog,[platformer],NaN,1992,Sonic the Hedgehog 2,894


In [14]:
import numpy as np

values = []
for row in new_games_df["genre"].tail(200):
    values.append(row)
unique, counts = np.unique(np.concatenate(values), return_counts=True)
pd.DataFrame({'val': unique, 'count': counts}).sort_values(by='count', ascending=False).head(50)

,val,count
4,action,71
6,adventure,48
55,rpg,27
22,fps,22
42,platform,19
71,survival,16
2,3rd person,15
26,horror,15
62,shooter,11
76,turn based,9


In [3]:
import numpy as np

values = []
for row in new_games_df["console_platform"]:
    values.append(row)
unique, counts = np.unique(np.concatenate(values), return_counts=True)
pd.DataFrame({'val': unique, 'count': counts}).sort_values(by='count', ascending=False)

,val,count
28,PC,565
35,PS4,186
46,XONE,150
45,X360,134
34,PS3,114
41,Switch,85
36,PS5,82
49,Xbox Series,79
33,PS2,63
23,Mobile,45


In [10]:
import numpy as np

unique, counts = np.unique(new_games_df["developer"], return_counts=True)
pd.DataFrame({'val': unique, 'count': counts}).sort_values(by='count', ascending=False)[50:100]

,val,count
43,Atlus,3
347,NetherRealm Studios,3
375,Obsidian Entertainment,3
351,Next Level Games,3
264,Japan Studio,3
338,Namco Limited,3
286,Lionhead Studios Ltd.,3
396,Platinum Games Inc.,3
388,Pandemic Studios,3
349,"Neversoft Entertainment, Inc.",3
